# CLIP detector with LayerNorm tuning

Follows GenD ([arXiv 2508.06248](https://arxiv.org/abs/2508.06248)): a CLIP ViT-L/14 image encoder where only the LayerNorm parameters train, L2-normalized 1024-dimensional features, a linear two-class head, and the loss cross entropy + 0.1 alignment + 0.5 uniformity (`face_decision_lab.metric_losses`). Batches keep each real photo together with the fakes made from it (`face_decision_lab.pairs`), and training images are augmented (flip, affine, blur, color jitter, JPEG).

`INIT_FROM = "gend_official"` starts from the authors' checkpoint trained on FaceForensics++ c23 (`yermandy/GenD_CLIP_L_14`, MIT) instead of CLIP as released, so training adds the lab's data to what that checkpoint already detects, and reframes every crop to the 1.3x face margin it was trained on (`tighten_crop`).

Departures from the paper: `EPOCHS` (paper: two cycles of ten) and the augmentation strengths, which the paper's summary does not give.

Each protocol trains a fresh model, checkpoints every epoch to a private Hub repository and resumes from it, so a lost Colab session costs at most one epoch. A protocol whose report is already on the Hub is skipped.

**Runtime:** Colab Pro, L4 or A100. **Needs** the Colab secret `HF_TOKEN` with write access.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-data"
HF_MODEL = "paulohenriquevn/face-decision-clip-ln"  # private
CLIP_MODEL = "openai/clip-vit-large-patch14"
# lab_ffpp_real trains the lab's photo expert with the FF++ originals as extra bona fide video, so
# it stops scoring real video as an attack; it also scores Celeb-DF and the FF++ test split.
# lab_ffpp_real_sr adds super-resolved copies of its real crops (build_superresolved.ipynb), to
# test whether that stops it flagging DiffusionFace's CelebA-HQ photos.
# ffpp_deepfakes_to_diff trains the lab's recipe on FF++ originals and DeepFakes alone and tests on
# DiFF, the protocol of the published DiFF results it is compared with (FSFM, FS-VFM).
PROTOCOL_NAMES = ("ffpp_deepfakes_to_diff",)
# "openai": CLIP as released. "gend_official": the GenD checkpoint trained on FaceForensics++ c23
# (yermandy/GenD_CLIP_L_14, MIT), so training adds the lab's data to what it already detects.
INIT_FROM = "openai"
# Face box margin the model sees, as a multiple of the box: GenD trained on 1.3x, the lab's crops
# hold 1.5x and tighten_crop reframes them. Keep 1.3 with INIT_FROM = "gend_official".
FACE_SCALE = 1.5
EPOCHS = 1
BATCH_SIZE = 32
# Largest piece of a pair group in a batch. An FF++ group (an original and its manipulations,
# 16 frames each) is 80 crops; whole, it fills a batch with one scene.
MAX_PAIR_GROUP = 10
PEAK_LEARNING_RATE = 3e-4
WARMUP_FRACTION = 0.1  # of all training steps
MIN_LEARNING_RATE = 1e-5
AUGMENT = False
# "gend": cross entropy + alignment + uniformity. "ce": cross entropy only, for ablations.
LOSS = "ce"
# Seed of the head's initial weights and of the batch order; a few seeds give the spread of a recipe.
RUN_SEED = 3407
# True scores a run whose report already exists again, from its last checkpoint and without
# training, to add what newer notebook versions report (saved scores, detection at BPCER).
RESCORE = True
RUN_TAG = f"{INIT_FROM}-{LOSS}-e{EPOCHS}-b{BATCH_SIZE}-lr{PEAK_LEARNING_RATE:g}-fs{FACE_SCALE:g}-g{MAX_PAIR_GROUP}" + ("" if AUGMENT else "-noaug") + ("" if RUN_SEED == 3407 else f"-s{RUN_SEED}")  # HF_MODEL/RUN_TAG/<protocol>/
# Other protocols' test sides the model trained for a protocol is also scored on: none of their
# crops are on its training side.
ALSO_SCORE = {
    "lab_ffpp_real": ("cross_celebdf", "ffpp"),
    "lab_ffpp_real_sr": ("cross_celebdf", "ffpp", "cross_diffusionface"),
    "lab_df40": ("cross_celebdf", "ffpp", "cross_diffusionface", "df40_ff"),
    "ffpp_deepfakes_to_diff": ("cross_celebdf", "ffpp"),
    "cross_celebdf": ("subject_holdout",),
    "ffpp_to_celebdf": ("ffpp",),
    "lab_ffpp_to_celebdf": ("subject_holdout", "ffpp"),
}
# Private datasets the external protocols load next to the lab data, by source_dataset. DiFF keeps
# its pristine images in a dataset of their own.
EXTERNAL_DATASET_REPOS = {
    "celebdf": ("paulohenriquevn/face-decision-celebdf-test",),
    "faceforensics": ("paulohenriquevn/face-decision-ffpp",),
    "diff": ("paulohenriquevn/face-decision-diff-test", "paulohenriquevn/face-decision-diff-pristine-test"),
    "diffusionface": ("paulohenriquevn/face-decision-diffusionface-test",),
    "superresolved": ("paulohenriquevn/face-decision-superresolved",),
    "df40": ("paulohenriquevn/face-decision-df40",),
}

### Installation

In [ ]:
%%capture
!pip install -q "face-decision-lab @ {REPO_SOURCE}"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Data

In [ ]:
from face_decision_lab.hub import load_hub_samples

from face_decision_lab.protocol import external_datasets

samples, hub_images = load_hub_samples(HF_DATASET)
images = {s.image_path: image for s, image in zip(samples, hub_images, strict = True)}
needed = list(PROTOCOL_NAMES) + [other for p in PROTOCOL_NAMES for other in ALSO_SCORE.get(p, ())]
for source in sorted(set().union(*(external_datasets(p) for p in needed))):
    for index, repo in enumerate(EXTERNAL_DATASET_REPOS[source]):
        extra, extra_images = load_hub_samples(repo, key_prefix = f"{source}{index}")
        samples += extra
        images.update({s.image_path: image for s, image in zip(extra, extra_images, strict = True)})
print(len(samples), "crops")

### Model and augmentation

In [ ]:
import io, math, random
import torch
from PIL import Image
from torch import nn
from torchvision import transforms
from transformers import CLIPImageProcessor, CLIPVisionModel

processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL)

def is_layer_norm(name):
    # CLIP vision names: pre_layrnorm, encoder.layers.N.layer_norm1/2, post_layernorm.
    return any(part in name for part in ("layer_norm", "layrnorm", "layernorm"))

GEND_OFFICIAL = "yermandy/GenD_CLIP_L_14"

class GenDDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = CLIPVisionModel.from_pretrained(CLIP_MODEL)
        # Non-reentrant checkpointing: the reentrant one drops gradients when the input needs none.
        self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs = {"use_reentrant": False})
        for name, parameter in self.encoder.named_parameters():
            parameter.requires_grad = is_layer_norm(name)
        self.head = nn.Linear(self.encoder.config.hidden_size, 2)
        if INIT_FROM == "gend_official":
            self.load_gend_official()

    def load_gend_official(self):
        # Same layout: CLIP vision pooler output, L2 normalization, a two-class linear head (index 1 = fake).
        from huggingface_hub import hf_hub_download
        from safetensors.torch import load_file
        state = load_file(hf_hub_download(GEND_OFFICIAL, "model.safetensors"))
        # transformers 4 names the vision parameters "vision_model.*"; transformers 5 drops the prefix.
        own = set(self.state_dict())
        renamed = {}
        for name, tensor in state.items():
            if name.startswith("feature_extractor.vision_model."):
                rest = name.removeprefix("feature_extractor.vision_model.")
                target = next((t for t in ("encoder.vision_model." + rest, "encoder." + rest) if t in own), None)
                assert target is not None, f"no parameter for {name}"
                renamed[target] = tensor
            elif name.startswith("model.linear."):
                renamed["head." + name.removeprefix("model.linear.")] = tensor
        loaded = self.load_state_dict(renamed, strict = False)
        assert not loaded.unexpected_keys, loaded.unexpected_keys[:5]
        missing = [n for n in loaded.missing_keys if "position_ids" not in n]
        assert not missing, missing[:5]

    def forward(self, pixels):
        features = nn.functional.normalize(self.encoder(pixel_values = pixels).pooler_output, dim = -1)
        return self.head(features), features

class RandomJpeg:
    def __init__(self, quality = (60, 95), p = 0.3):
        self.quality, self.p = quality, p

    def __call__(self, image):
        if random.random() >= self.p:
            return image
        buffer = io.BytesIO()
        image.save(buffer, format = "JPEG", quality = random.randint(*self.quality))
        return Image.open(io.BytesIO(buffer.getvalue())).convert("RGB")

augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.RandomAffine(degrees = 5, translate = (0.05, 0.05), scale = (0.95, 1.05))], p = 0.5),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma = (0.1, 1.5))], p = 0.2),
    transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.2, 0.02)], p = 0.5),
    RandomJpeg(),
])

def warmup_cosine(steps_per_epoch, epochs):
    # Linear warm-up from MIN to PEAK over WARMUP_FRACTION of the steps, then cosine decay back to MIN.
    total = steps_per_epoch * epochs
    warmup = max(1, round(total * WARMUP_FRACTION))
    low = MIN_LEARNING_RATE / PEAK_LEARNING_RATE
    def factor(step):
        if step < warmup:
            return low + (1 - low) * step / max(1, warmup)
        progress = (step - warmup) / max(1, total - warmup)
        return low + (1 - low) * 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
    return factor

probe = GenDDetector()
trainable = sum(p.numel() for p in probe.parameters() if p.requires_grad)
print(f"trainable {trainable:,} of {sum(p.numel() for p in probe.parameters()):,}")
if INIT_FROM == "gend_official":
    # The model card's examples: an FF++ Deepfakes frame, then a real frame.
    import requests
    examples = ["https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/DF/000_003/000.png?raw=true",
                "https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/real/000/000.png?raw=true"]
    frames = [Image.open(io.BytesIO(requests.get(u, timeout = 30).content)).convert("RGB") for u in examples]
    with torch.no_grad():
        logits, _ = probe.eval()(processor(images = frames, return_tensors = "pt")["pixel_values"])
    fake_probability = logits.softmax(-1)[:, 1].tolist()
    print("official checkpoint on its examples (fake, real):", fake_probability)
    assert fake_probability[0] > 0.5 > fake_probability[1], fake_probability
del probe

### Train with per-epoch checkpoints, then score

The report on the protocol's own test side goes to `HF_MODEL/RUN_TAG/<protocol>/report.json`; those on the `ALSO_SCORE` test sides go next to it, as `<other protocol>.json`. Every report holds the detection at BPCER 0.1%, 0.2% and 0.5% with the threshold fixed on the protocol's calibration side (`detection_at_bpcer`), and every crop's score goes to `scores/<side>.npz` in the same folder (`score_files.save_scores`): `calibration`, `test` and one file per `ALSO_SCORE` protocol.

In [ ]:
import json, time
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm
from face_decision_lab.metric_losses import gend_loss
from face_decision_lab.pairs import paired_batches
from face_decision_lab.protocol import auc_by_group, auc_within_groups, detection_at_bpcer, evaluation_report, protocol_split
from face_decision_lab.score_files import save_scores
from face_decision_lab.sources import df40, diff
from face_decision_lab.faces import tighten_crop
from face_decision_lab.rows import open_image

api = HfApi()
api.create_repo(HF_MODEL, private = True, exist_ok = True)
pool = ThreadPoolExecutor(max_workers = 8)

def pixels(part, train):
    def load(sample):
        image = tighten_crop(open_image(images[sample.image_path]), FACE_SCALE)
        return augment(image) if train and AUGMENT else image
    return processor(images = list(pool.map(load, part)), return_tensors = "pt")["pixel_values"]

def prefetched(batches, train):
    with ThreadPoolExecutor(max_workers = 1) as loader:
        future = loader.submit(pixels, batches[0], train)
        for index in range(len(batches)):
            current = future.result()
            if index + 1 < len(batches):
                future = loader.submit(pixels, batches[index + 1], train)
            yield batches[index], current

def hub_has(path):
    return path in set(api.list_repo_files(HF_MODEL))

def score_crops(model, part):
    values = []
    with torch.no_grad(), torch.autocast("cuda", dtype = torch.bfloat16):
        for _batch, x in prefetched([part[i:i + 128] for i in range(0, len(part), 128)], False):
            logits, _ = model(x.cuda())
            values += torch.softmax(logits.float(), dim = -1)[:, 1].cpu().tolist()
    return values

def upload_scores(folder, local, side, part, values):
    (local / "scores").mkdir(exist_ok = True)
    save_scores(local / "scores" / f"{side}.npz", part, {"model": np.array(values)})
    api.upload_file(path_or_fileobj = str(local / "scores" / f"{side}.npz"), path_in_repo = f"{folder}/scores/{side}.npz", repo_id = HF_MODEL, commit_message = f"{folder}/scores/{side}")

def train_and_score(protocol):
    folder = f"{RUN_TAG}/{protocol}"
    local = Path(f"/content/clip_gend/{protocol}")
    local.mkdir(parents = True, exist_ok = True)
    train, calibration, test = protocol_split(samples, protocol)
    torch.manual_seed(RUN_SEED)
    model = GenDDetector().cuda()
    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr = PEAK_LEARNING_RATE)
    epochs = [paired_batches(train, BATCH_SIZE, RUN_SEED + epoch, max_group = MAX_PAIR_GROUP) for epoch in range(EPOCHS)]
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_cosine(len(epochs[0]), EPOCHS))
    start = 0
    if hub_has(f"{folder}/checkpoint.pt"):
        state = torch.load(hf_hub_download(HF_MODEL, f"{folder}/checkpoint.pt", force_download = True), weights_only = False)
        model.load_state_dict(state["trainable"], strict = False)
        optimizer.load_state_dict(state["optimizer"]); scheduler.load_state_dict(state["scheduler"])
        start = state["epoch"] + 1
        print(f"{protocol}: resuming after epoch {state['epoch']}")
    started = time.time()
    for epoch in range(start, EPOCHS):
        model.train()
        for step, (batch, x) in enumerate(tqdm(prefetched(epochs[epoch], True), total = len(epochs[epoch]), desc = f"{protocol} epoch {epoch}")):
            y = torch.tensor([0 if s.is_bona_fide else 1 for s in batch], device = "cuda")
            with torch.autocast("cuda", dtype = torch.bfloat16):
                logits, features = model(x.cuda())
            loss = gend_loss(logits.float(), features.float(), y) if LOSS == "gend" else nn.functional.cross_entropy(logits.float(), y)
            loss.backward()
            if step == 0:
                missing = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
                assert not missing, f"no gradient reached {missing[:3]}"
            optimizer.step(); optimizer.zero_grad(); scheduler.step()
            if step % 100 == 0:
                print(f"epoch {epoch} step {step} loss {loss.item():.4f} lr {scheduler.get_last_lr()[0]:.2e}")
        trainable_state = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
        torch.save({"trainable": trainable_state, "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "epoch": epoch}, local / "checkpoint.pt")
        api.upload_file(path_or_fileobj = str(local / "checkpoint.pt"), path_in_repo = f"{folder}/checkpoint.pt", repo_id = HF_MODEL, commit_message = f"{folder} epoch {epoch}")
    model.eval()
    calibration_scores = score_crops(model, calibration)
    upload_scores(folder, local, "calibration", calibration, calibration_scores)
    scores = score_crops(model, test)
    upload_scores(folder, local, "test", test, scores)
    for other in ALSO_SCORE.get(protocol, ()):
        _, _, other_test = protocol_split(samples, other)
        other_scores = score_crops(model, other_test)
        upload_scores(folder, local, other, other_test, other_scores)
        other_report = evaluation_report(other_test, other_scores) | {"trained_for": protocol}
        other_report["detection_at_bpcer"] = detection_at_bpcer(calibration, calibration_scores, other_test, other_scores)
        if other == "df40_ff":  # each unknown method against its own reals
            other_report["auc_by_unknown_method"] = auc_within_groups(other_test, other_scores, df40.unknown_group)
        api.upload_file(path_or_fileobj = json.dumps(other_report, indent = 2).encode(), path_in_repo = f"{folder}/{other}.json", repo_id = HF_MODEL, commit_message = f"{folder}/{other} report")
        print(other, json.dumps({k: other_report[k] for k in ("auc", "eer", "bpcer_at_max_apcer")}))
    report = evaluation_report(test, scores)
    report["detection_at_bpcer"] = detection_at_bpcer(calibration, calibration_scores, test, scores)
    if {s.source_dataset for s in test} == {"diff"}:  # DiFF's published results: one AUC per subset
        report["auc_by_subset"] = auc_by_group(test, scores, diff.subset)
    report.update(training_rows = len(train), epochs = EPOCHS, batch_size = BATCH_SIZE, augment = AUGMENT, init_from = INIT_FROM, loss = LOSS, run_seed = RUN_SEED, train_minutes_this_session = round((time.time() - started) / 60, 1))
    (local / "report.json").write_text(json.dumps(report, indent = 2))
    api.upload_file(path_or_fileobj = str(local / "report.json"), path_in_repo = f"{folder}/report.json", repo_id = HF_MODEL, commit_message = f"{folder} report")
    return report

reports = {}
for protocol in PROTOCOL_NAMES:
    if hub_has(f"{RUN_TAG}/{protocol}/report.json") and not RESCORE:
        print(protocol, "already scored; skipping")
        continue
    reports[protocol] = train_and_score(protocol)
    print(protocol, json.dumps({k: reports[protocol][k] for k in ("auc", "eer", "bpcer_at_max_apcer", "training_rows")}))